# Notebook 1 +

## Задание 1

Из БВ FRED загрузите ряд `AAA` ([ссылка](https://fred.stlouisfed.org/series/AAA)) c **2000-01-01** по **2025-12-31**.Пусть `y` - первая разность это ряда. Для ряда `y` выберите оптимальный порядок модели AR(L)-GARCH (p,o,q) с распределением для нормализованных шоков из списка: **['normal', 't', 'skewt']** Используйте метод кросс-валидации: **расширяющаяся** обучающая выборка. Используйте следующие параметры кросс-валидации: длина (первоначальной) обучающей выборки - 100, горизонт прогнозирования - 5 периодов, сдвиг - 5 периодов. Метрика кросс-валидации **MSE**. Рассмотрите следующие диапазоны: 0≤L≤2, 1≤p≤2, 0≤o≤2, 1≤q≤2. Остальные параметры модели по умолчанию.

**Указание**: не забудьте задать правильный периодический индекс для ряда `y`

L=Ответ {$а} Вопрос 1

p=Ответ {$а} Вопрос 1

o=Ответ {$а} Вопрос 1

q=Ответ {$а} Вопрос 1

Распределение шоков: Ответ {$а} Вопрос 1 normal t skewt

In [1]:
import warnings
warnings.filterwarnings("ignore")

import itertools
import numpy as np
import pandas as pd
from pandas_datareader.data import DataReader
from arch import arch_model

# 1. Загружаем ряд AAA из FRED
x = DataReader(
    "AAA",
    "fred",
    start="2000-01-01",
    end="2025-12-31",
)["AAA"].dropna()

# Важно: AAA - месячный ряд, задаем периодический индекс
x.index = x.index.to_period("M")

# 2. Первая разность
y = x.diff().dropna()

# 3. Параметры CV
initial_train = 100
horizon = 5
step = 5

Ls = range(0, 3)
ps = range(1, 3)
os = range(0, 3)
qs = range(1, 3)
dists = ["normal", "t", "skewt"]

results = []

for L, p, o, q, dist in itertools.product(Ls, ps, os, qs, dists):
    errors = []

    for train_end in range(initial_train, len(y) - horizon + 1, step):
        y_train = y.iloc[:train_end]
        y_test = y.iloc[train_end:train_end + horizon]

        model = arch_model(
            y_train,
            mean="ARX",
            lags=L,
            vol="GARCH",
            p=p,
            o=o,
            q=q,
            dist=dist,
        )

        res = model.fit(disp="off")
        forecast = res.forecast(horizon=horizon, reindex=False)

        y_pred = forecast.mean.iloc[-1].to_numpy()
        errors.extend((y_test.to_numpy() - y_pred) ** 2)

    mse = np.mean(errors)

    results.append({
        "L": L,
        "p": p,
        "o": o,
        "q": q,
        "dist": dist,
        "mse": mse,
    })

cv_results = pd.DataFrame(results).sort_values("mse")
print(cv_results.head(10))

best = cv_results.iloc[0]
print(best)


    L  p  o  q    dist       mse
9   0  1  1  2  normal  0.038637
6   0  1  1  1  normal  0.038637
33  0  2  2  2  normal  0.038644
30  0  2  2  1  normal  0.038644
23  0  2  0  2   skewt  0.038651
20  0  2  0  1   skewt  0.038651
2   0  1  0  1   skewt  0.038660
5   0  1  0  2   skewt  0.038660
21  0  2  0  2  normal  0.038668
18  0  2  0  1  normal  0.038668
L              0
p              1
o              1
q              2
dist      normal
mse     0.038637
Name: 9, dtype: object
